PRINCIPAL

In [ ]:
!pip install scikit-optimize

SALVAR MODELO

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from skopt.utils import use_named_args
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
import os
import joblib
import pickle
from datetime import datetime
import json
warnings.filterwarnings('ignore')

# ============================================================================
# FUNÇÃO PARA CONVERTER TIPOS NUMPY PARA PYTHON
# ============================================================================

def converter_para_python(obj):
    """Converte objetos numpy para tipos Python nativos para serialização JSON"""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: converter_para_python(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [converter_para_python(v) for v in obj]
    elif isinstance(obj, tuple):
        return tuple(converter_para_python(v) for v in obj)
    else:
        return obj

# ============================================================================
# CONFIGURAÇÕES
# ============================================================================

# ESCOLHA A AÇÃO QUE DESEJA ANALISAR
TICKER_ESCOLHIDO = 'BOVA11.SA'  # <- ALTERE AQUI para qualquer ticker da lista
NOME_ATIVO = 'BOVA11.SA'  # <- ALTERE AQUI o nome descritivo

# ⚙️ CONFIGURAÇÕES DE ESTRATÉGIA
HABILITAR_VENDA = False  # True = Compra/Venda/Neutro | False = Compra/Neutro apenas
USAR_NEUTRO = True  # True = Inclui sinais neutros | False = Só Compra/Venda

# ⚙️ CONFIGURAÇÕES DE ALAVANCAGEM
ALAVANCAGEM_ML = 1 # Alavancagem para estratégia ML (1 = sem alavancagem, 2 = 2x)
ALAVANCAGEM_BH = 1  # Alavancagem para Buy & Hold (normalmente 1)

# 💰 CONFIGURAÇÕES DE CUSTOS OPERACIONAIS
CUSTOS = {
    'corretagem': 0.0005,        # 0.05% por trade (ex: Clear, Inter)
    'slipage': 0.001,           # 0.1% de slippage (derrapagem)
    'emolumentos': 0.00003,     # 0.003% taxa B3
    'taxa_liquidacao': 0.000025, # 0.0025% taxa de liquidação
    'iss': 0.00005,             # 0.005% ISS (apenas SP)
    'custo_total_por_trade': 0  # Será calculado automaticamente
}

# Calcular custo total por trade (ida e volta)
CUSTOS['custo_total_por_trade'] = (
    CUSTOS['corretagem'] * 2 +      # Ida e volta
    CUSTOS['slipage'] * 2 +         # Slippage na entrada e saída
    CUSTOS['emolumentos'] * 2 +     # B3 na compra e venda
    CUSTOS['taxa_liquidacao'] * 2 + # Liquidação na compra e venda
    CUSTOS['iss']                   # ISS apenas na venda
)

# Lista completa de tickers disponíveis (para referência)
TICKERS_DISPONIVEIS = {
    # ETFs
    'BOVA11.SA':'IBOV',
    'IVVB11.SA':'SP500',
    'SMAL11.SA':'SMALL',
    'DIVO11.SA':'DIVIDENDOS',
    'PIBB11.SA':'IBRX50',
    'XBOV11.SA':'IBOV2',

    # FIIs Papel
    'CPTS11.SA':'CPTS',
    'VGIR11.SA':'VGIR',
    'KNCR11.SA':'KNCR',
    'IRDM11.SA':'IRDM',
    'RBRR11.SA':'RBRR',
    'MXRF11.SA':'MXRF',

    # FIIs Logísticos
    'XPLG11.SA':'XPLG',
    'HGLG11.SA':'HGLG',
    'BTLG11.SA':'BTLG',
    'LVBI11.SA':'LVBI',
    'BRCO11.SA':'BRCO',
    'VISC11.SA':'VISC',

    # Energia
    'TAEE11.SA':'TAESA',
    'EGIE3.SA':'ENGIE',
    'CMIG4.SA':'CEMIG',
    'ALUP11.SA':'ALUPAR',

    # Bancos
    'ITUB4.SA':'ITAU',
    'BBAS3.SA':'BB',
    'BPAC11.SA':'BTG',
    'ITSA4.SA':'ITAUSA',
    'SANB11.SA':'SANTANDER',
    'B3SA3.SA':'B3'
}

TRY = 30 # Número de iterações da otimização bayesiana
INVESTIMENTO_INICIAL = 10000.00  # Investimento inicial para ação única
FUT = 3

inferior = 0.35
superior = 0.65

N_SPLITS_OUTER = 5
N_SPLITS_INNER = 3
N_SPLITS_FINAL = 3

# ============================================================================
# FUNÇÕES AUXILIARES
# ============================================================================

def get_walk_forward_splits(df, n_splits):
    """Retorna lista de (treino, validação) para walk-forward"""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    splits = []
    for idx_treino, idx_val in tscv.split(df):
        splits.append((idx_treino, idx_val))
    return splits

def gerar_sinais_trading(predicoes, habilitar_venda=True, usar_neutro=True):
    """
    Converte previsões do modelo em sinais de trading

    Parâmetros:
    - predicoes: array com classes previstas (0, 1, 2)
    - habilitar_venda: se True, classe 0 = Venda (-1)
    - usar_neutro: se True, classe 1 = Neutro (0)

    Retorna:
    - array de sinais: -1 (venda), 0 (neutro), 1 (compra)
    """
    sinais = np.zeros(len(predicoes), dtype=int)

    # Classe 2 = Compra (sempre habilitada)
    sinais[predicoes == 2] = 1

    # Classe 1 = Neutro (se habilitado)
    if usar_neutro:
        sinais[predicoes == 1] = 0
    else:
        # Se não usa neutro, transforma classe 1 em 0 (fora do mercado)
        sinais[predicoes == 1] = 0

    # Classe 0 = Venda (se habilitado)
    if habilitar_venda:
        sinais[predicoes == 0] = -1
    else:
        # Se venda desabilitada, transforma classe 0 em 0 (fora do mercado)
        sinais[predicoes == 0] = 0

    return sinais

def aplicar_custos_operacionais(df_trades, custos):
    """
    Aplica custos operacionais realistas nos retornos

    Parâmetros:
    - df_trades: DataFrame com sinais e retornos
    - custos: dicionário com custos configurados

    Retorna:
    - DataFrame com custos aplicados
    """
    df = df_trades.copy()

    # Identificar mudanças de sinal (trades)
    df['Mudanca_Sinal'] = df['Sinal'].diff().abs() > 0
    df['Trade_Entrada'] = df['Mudanca_Sinal'] & (df['Sinal'] != 0)
    df['Trade_Saida'] = df['Mudanca_Sinal'] & (df['Sinal'].shift(1) != 0)

    # Inicializar coluna de custos
    df['Custo_Operacao'] = 0.0
    df['Custo_Acumulado'] = 0.0
    custo_acumulado = 0.0

    for i in range(len(df)):
        if df.iloc[i]['Trade_Entrada'] or df.iloc[i]['Trade_Saida']:
            # Aplicar custo total do trade
            preco_atual = df.iloc[i]['Close']
            custo_trade = preco_atual * custos['custo_total_por_trade']

            # Se for apenas entrada (não houve saída), cobra metade
            if df.iloc[i]['Trade_Entrada'] and not df.iloc[i]['Trade_Saida']:
                custo_trade = custo_trade / 2

            # Se for apenas saída (não houve entrada), cobra metade
            if df.iloc[i]['Trade_Saida'] and not df.iloc[i]['Trade_Entrada']:
                custo_trade = custo_trade / 2

            df.iloc[i, df.columns.get_loc('Custo_Operacao')] = custo_trade
            custo_acumulado += custo_trade

        df.iloc[i, df.columns.get_loc('Custo_Acumulado')] = custo_acumulado

    # Ajustar retorno da estratégia com custos
    df['Retorno_ML_Bruto'] = df['Retorno_ML'].copy()
    df['Retorno_ML'] = df['Retorno_ML'] - (df['Custo_Operacao'] / (INVESTIMENTO_INICIAL * ALAVANCAGEM_ML))

    # Também aplicar custo inicial no Buy & Hold (apenas na compra inicial)
    if len(df) > 0:
        custo_inicial_bh = df.iloc[0]['Close'] * custos['custo_total_por_trade'] / 2
        df['Retorno_BH'] = df['Retorno_BH'] - (custo_inicial_bh / INVESTIMENTO_INICIAL)

    return df

def descrever_estrategia(habilitar_venda, usar_neutro, alavancagem_ml, alavancagem_bh, custos):
    """Retorna descrição textual da estratégia configurada"""
    if habilitar_venda and usar_neutro:
        estrategia = "Compra/Venda/Neutro"
    elif habilitar_venda and not usar_neutro:
        estrategia = "Compra/Venda"
    elif not habilitar_venda and usar_neutro:
        estrategia = "Compra/Neutro"
    else:
        estrategia = "Apenas Compra"

    custo_total_pct = custos['custo_total_por_trade'] * 100

    return f"{estrategia} (ML: {alavancagem_ml}x | B&H: {alavancagem_bh}x | Custos: {custo_total_pct:.2f}%/trade)"

def calcular_features_sem_target(dados, roll_curto, roll_longo):
    """
    Calcula todas as features exceto o target (Alvo).
    O target será calculado separadamente usando os quantis do treino.
    """
    df = dados[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()

    # =========================
    # RETORNOS
    # =========================
    df['ret_1']  = df['Close'].pct_change(1)
    df['ret_5']  = df['Close'].pct_change(5)
    df['ret_10'] = df['Close'].pct_change(roll_curto)
    df['ret_21'] = df['Close'].pct_change(roll_longo)

    # =========================
    # MÉDIAS
    # =========================
    df['mm10'] = df['Close'].rolling(roll_curto).mean()
    df['mm21'] = df['Close'].rolling(roll_longo).mean()

    # =========================
    # VOLATILIDADE
    # =========================
    df['vol_10']  = df['ret_1'].rolling(roll_curto).std()
    df['vol_21']  = df['ret_1'].rolling(roll_longo).std()
    df['vol_norm'] = df['vol_10'] / (df['vol_21'] + 1e-6)

    # =========================
    # DRIVER
    # =========================
    df['driver_sharpe'] = (
        df['ret_10'].rolling(roll_curto).mean() /
        (df['ret_10'].rolling(roll_curto).std() + 1e-4)
    )

    ewm   = df['Close'].ewm(span=roll_curto, adjust=False).mean()
    ruido = df['Close'] - ewm

    df['driver_ewm'] = (
        ewm.pct_change() /
        (ruido.rolling(roll_curto).std() + 1e-4)
    )

    df['roc_suave'] = df['ret_10'].rolling(5).mean()

    df['driver_roc'] = (
        df['roc_suave'] /
        (df['ret_10'].rolling(roll_curto).std() + 1e-4)
    )

    df['driver'] = (
        df['driver_sharpe'] * 0.40 +
        df['driver_ewm'] * 0.30 +
        df['driver_roc'] * 0.30
    )

    df['driver'] = np.clip(df['driver'], -3, 3)

    # =========================
    # CONTADOR DE TENDÊNCIA
    # =========================
    df['cont'] = 0
    cont_plus = cont_down = 0

    for i in df.index:
        if df.loc[i, 'mm10'] < df.loc[i, 'Close']:
            cont_plus += 1
            cont_down = 0
        elif df.loc[i, 'mm10'] > df.loc[i, 'Close']:
            cont_down -= 1
            cont_plus = 0
        else:
            cont_plus = cont_down = 0

        df.loc[i, 'cont'] = cont_plus + cont_down

    df['driver_mom']   = df['driver'].diff(5)
    df['driver_suave'] = df['driver'].rolling(5).mean()
    df['driver_conf']  = df['driver'].abs()

    df['vol_zscore'] = (
        (df['Volume'] - df['Volume'].rolling(roll_curto).mean()) /
        (df['Volume'].rolling(roll_curto).std() + 1e-6)
    )

    df['preco_vs_mm10'] = df['Close'] / df['mm10'] - 1
    df['dist_max_10'] = df['Close'] / df['High'].rolling(roll_curto).max() - 1

    # ==========================================================
    # NOVAS FEATURES
    # ==========================================================

    high_roll = df['High'].rolling(roll_curto).max()
    low_roll = df['Low'].rolling(roll_curto).min()

    # Eficiência da tendência
    mov_liquido = (df['Close'] - df['Close'].shift(roll_curto)).abs()
    mov_total = df['Close'].diff().abs().rolling(roll_curto).sum()
    df['trend_eff'] = mov_liquido / (mov_total + 1e-6)

    # Posição na faixa
    df['range_pos'] = (
        (df['Close'] - low_roll) /
        (high_roll - low_roll + 1e-6)
    )

    # Distância do topo
    df['dist_topo'] = (
        (high_roll - df['Close']) /
        (df['vol_10'] * df['Close'] + 1e-6)
    )

    # Distância do fundo
    df['dist_fundo'] = (
        (df['Close'] - low_roll) /
        (df['vol_10'] * df['Close'] + 1e-6)
    )

    # Inclinação da média
    df['mm10_slope'] = (
        (df['mm10'] - df['mm10'].shift(5)) /
        (df['vol_10'] * df['Close'] + 1e-6)
    )

    # Razão entre médias
    df['mm_ratio'] = df['mm10'] / (df['mm21'] + 1e-6)

    # Corpo do candle
    df['body_ratio'] = (
        (df['Close'] - df['Open']) /
        (df['High'] - df['Low'] + 1e-6)
    )

    # Pressão compradora
    df['buy_pressure'] = (
        (df['Close'] - df['Low']) /
        (df['High'] - df['Low'] + 1e-6)
    )

    # Pressão vendedora
    df['sell_pressure'] = (
        (df['High'] - df['Close']) /
        (df['High'] - df['Low'] + 1e-6)
    )

    # Gap
    df['gap'] = (
        (df['Open'] - df['Close'].shift(1)) /
        (df['vol_10'] * df['Close'] + 1e-6)
    )

    # Aceleração do retorno
    df['ret_acc'] = df['ret_5'] - df['ret_21']

    # Mudança da volatilidade
    df['vol_change'] = df['vol_norm'].diff(5)

    # Persistência da tendência
    df['trend_persist'] = (
        np.sign(df['ret_1'])
        .rolling(roll_curto)
        .mean()
    )

    # Assimetria
    df['ret_skew'] = (
        df['ret_1']
        .rolling(roll_curto)
        .skew()
    )

    # Curtose
    df['ret_kurt'] = (
        df['ret_1']
        .rolling(roll_curto)
        .kurt()
    )

    # Momentum ajustado pelo drawdown
    ret = df['Close'] / df['Close'].shift(roll_curto) - 1

    draw = (
        df['Close'] /
        df['Close'].rolling(roll_curto).max()
        - 1
    )

    df['mom_draw'] = ret / (draw.abs() + 0.01)

    # Volatilidade direcional
    up = (
        df['ret_1']
        .clip(lower=0)
        .rolling(roll_curto)
        .std()
    )

    down = (
        df['ret_1']
        .clip(upper=0)
        .rolling(roll_curto)
        .std()
    )

    df['vol_direction'] = up / (down + 1e-6)

    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    return df

FEATURES = [
    'ret_1','ret_5','ret_10','ret_21',
    'driver','driver_mom','driver_suave','driver_conf',
    'cont','vol_10','vol_norm','vol_zscore',
    'preco_vs_mm10','dist_max_10',

    'trend_eff',
    'range_pos',
    'dist_topo',
    'dist_fundo',
    'mm10_slope',
    'mm_ratio',
    'body_ratio',
    'buy_pressure',
    'sell_pressure',
    'gap',
    'ret_acc',
    'vol_change',
    'trend_persist',
    'ret_skew',
    'ret_kurt',
    'mom_draw',
    'vol_direction'
]

# ============================================================================
# ESPAÇO DE HIPERPARÂMETROS (12 DIMENSÕES)
# ============================================================================

espaco_parametros = [
    Categorical(['10y'], name='time_period'),
    Integer(5, 20, name='roll_curto'),
    Integer(16, 40, name='roll_longo'),

    # XGBoost
    Integer(50, 300, name='n_estimators'),
    Integer(2, 5, name='max_depth'),
    Real(0.01, 0.10, name='learning_rate'),
    Integer(5, 25, name='min_child_weight'),
    Real(1.0, 50.0, name='reg_alpha'),
    Real(5.0, 100.0, name='reg_lambda'),
    Real(0.1, 2.0, name='gamma'),
    Real(0.40, 0.80, name='subsample'),
    Real(0.30, 0.70, name='colsample_bytree'),
]

# ============================================================================
# FUNÇÃO OBJETIVO PARA UMA ÚNICA AÇÃO
# ============================================================================

@use_named_args(espaco_parametros)
def avaliar_configuracao_acao_unica(**params):
    """Avalia uma configuração para UMA ÚNICA ação"""
    time_period = params['time_period']
    roll_curto = int(params['roll_curto'])
    roll_longo = int(params['roll_longo'])

    # Validações
    if roll_curto >= roll_longo:
        return 5.0
    if roll_longo < roll_curto * 2:
        return 4.0

    try:
        # Baixar dados
        raw = yf.Ticker(TICKER_ESCOLHIDO).history(period=time_period, interval="1d").reset_index()
        if len(raw) < 252:
            return 3.0

        # Calcular features sem target
        df = calcular_features_sem_target(raw, roll_curto, roll_longo)
        if len(df) < 100:
            return 2.0

        base = XGBClassifier(
            n_estimators    = int(params['n_estimators']),
            max_depth       = int(params['max_depth']),
            learning_rate   = params['learning_rate'],
            min_child_weight= int(params['min_child_weight']),
            reg_alpha       = params['reg_alpha'],
            reg_lambda      = params['reg_lambda'],
            gamma           = params['gamma'],
            subsample       = params['subsample'],
            colsample_bytree= params['colsample_bytree'],
            random_state    = 42,
            objective='multi:softprob',
            num_class=3,
            eval_metric='mlogloss',
            verbosity       = 0
        )

        todos_retornos = []

        splits = get_walk_forward_splits(df, N_SPLITS_OUTER)

        for idx_treino, idx_val in splits:
            if len(idx_treino) < 50 or len(idx_val) < 20:
                continue

            # Separar treino e validação
            df_treino = df.iloc[idx_treino].copy()
            df_val = df.iloc[idx_val].copy()

            # Calcular quantis APENAS no treino
            driver_fut_treino = df_treino['driver'].shift(-FUT) - df_treino['driver']
            limite_venda = driver_fut_treino.quantile(inferior)
            limite_compra = driver_fut_treino.quantile(superior)

            # Criar labels no treino
            df_treino['Alvo'] = 1
            df_treino.loc[driver_fut_treino >= limite_compra, 'Alvo'] = 2
            df_treino.loc[driver_fut_treino <= limite_venda, 'Alvo'] = 0

            # Criar labels na validação usando MESMOS quantis
            driver_fut_val = df_val['driver'].shift(-FUT) - df_val['driver']
            df_val['Alvo'] = 1
            df_val.loc[driver_fut_val >= limite_compra, 'Alvo'] = 2
            df_val.loc[driver_fut_val <= limite_venda, 'Alvo'] = 0

            # Remover NaNs
            df_treino = df_treino.dropna(subset=['Alvo'])
            df_val = df_val.dropna(subset=['Alvo'])

            if len(df_treino) < 30 or len(df_val) < 10:
                continue

            X_treino = df_treino[FEATURES]
            y_treino = df_treino['Alvo']
            X_val    = df_val[FEATURES]
            y_val    = df_val['Alvo']

            tscv_calib = TimeSeriesSplit(n_splits=min(N_SPLITS_INNER, 3))
            modelo = CalibratedClassifierCV(base, method='isotonic', cv=tscv_calib)
            modelo.fit(X_treino, y_treino)

            sinal_pred = modelo.predict(X_val)

            # Usar a função gerar_sinais_trading com as configurações atuais
            sinal_trading = gerar_sinais_trading(sinal_pred, HABILITAR_VENDA, USAR_NEUTRO)

            val_data = df_val.copy()
            val_data['Sinal'] = sinal_trading
            val_data['Retorno_Real'] = val_data['Close'].pct_change()
            # Aplicar alavancagem no retorno da estratégia
            val_data['Retorno_ML'] = val_data['Sinal'].shift(1) * val_data['Retorno_Real'] * ALAVANCAGEM_ML
            val_data['Retorno_BH'] = val_data['Retorno_Real'] * ALAVANCAGEM_BH

            # Aplicar custos operacionais
            val_data = aplicar_custos_operacionais(val_data, CUSTOS)

            todos_retornos.append(val_data['Retorno_ML'].dropna())

        if not todos_retornos:
            return 1.0

        todos_retornos = pd.concat(todos_retornos)
        retorno_medio = todos_retornos.mean()
        std_retorno = todos_retornos.std()

        if std_retorno > 0:
            sharpe = (retorno_medio / std_retorno) * np.sqrt(252)
        else:
            sharpe = -5

        # Penalidade por poucos sinais (ajustada para considerar custos)
        num_sinais = len(todos_retornos[todos_retornos != 0])
        total_periodos = len(todos_retornos)
        taxa_sinal = num_sinais / total_periodos if total_periodos > 0 else 0

        if taxa_sinal < 0.10:
            sharpe -= 2.0
        elif taxa_sinal > 0.90:
            sharpe -= 1.0

        print(f"  📊 Sharpe: {sharpe:.3f} | Sinais: {taxa_sinal:.1%} | "
              f"Time: {time_period}, RC: {roll_curto}, RL: {roll_longo}")
        return -sharpe

    except Exception as e:
        print(f"  ❌ Erro: {str(e)[:100]}")
        return 10.0

# ============================================================================
# FUNÇÃO PARA SALVAR O MODELO
# ============================================================================

def salvar_modelo_acao_unica(modelo_final, ticker, nome_ativo):
    """Salva o modelo de uma única ação"""

    os.makedirs('modelos', exist_ok=True)
    os.makedirs('outputs', exist_ok=True)
    os.makedirs('backup', exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Nome do arquivo inclui configuração da estratégia
    estrategia_tag = ""
    if HABILITAR_VENDA and USAR_NEUTRO:
        estrategia_tag = "COMPLETO"
    elif HABILITAR_VENDA and not USAR_NEUTRO:
        estrategia_tag = "COMPRA_VENDA"
    elif not HABILITAR_VENDA and USAR_NEUTRO:
        estrategia_tag = "COMPRA_NEUTRO"
    else:
        estrategia_tag = "APENAS_COMPRA"

    # Adicionar informação de alavancagem e custos
    estrategia_tag += f"_ALAV{ALAVANCAGEM_ML}x_CUSTOS"

    nome_base = f"modelo_{nome_ativo}_{estrategia_tag}"

    print(f"\n{'='*72}")
    print(f"  SALVANDO MODELO - {nome_ativo} ({ticker})")
    print(f"  Estratégia: {descrever_estrategia(HABILITAR_VENDA, USAR_NEUTRO, ALAVANCAGEM_ML, ALAVANCAGEM_BH, CUSTOS)}")
    print(f"{'='*72}")

    # 1. Salvar modelo principal
    caminho_pkl = f'modelos/{nome_base}_{timestamp}.pkl'
    joblib.dump(modelo_final, caminho_pkl)
    print(f"✅ Modelo principal: {caminho_pkl}")

    # 2. Salvar versão simplificada
    modelo_simplificado = {
        'modelo': modelo_final['modelo'],
        'configuracao': modelo_final['configuracao'],
        'ticker': modelo_final['ticker'],
        'nome_ativo': modelo_final['nome_ativo']
    }
    caminho_simples = f'modelos/{nome_base}_simples_{timestamp}.pkl'
    joblib.dump(modelo_simplificado, caminho_simples)
    print(f"✅ Modelo simplificado: {caminho_simples}")

    # 3. Salvar configurações em JSON
    config_json = {
        'timestamp': timestamp,
        'data_criacao': datetime.now().isoformat(),
        'ticker': ticker,
        'nome_ativo': nome_ativo,
        'estrategia': {
            'habilitar_venda': HABILITAR_VENDA,
            'usar_neutro': USAR_NEUTRO,
            'alavancagem_ml': ALAVANCAGEM_ML,
            'alavancagem_bh': ALAVANCAGEM_BH,
            'custos': CUSTOS,
            'descricao': descrever_estrategia(HABILITAR_VENDA, USAR_NEUTRO, ALAVANCAGEM_ML, ALAVANCAGEM_BH, CUSTOS)
        },
        'configuracao': converter_para_python(modelo_final['configuracao']),
        'metricas_backtest': converter_para_python(modelo_final['metricas_backtest'])
    }
    caminho_json = f'modelos/config_{nome_ativo}_{estrategia_tag}_{timestamp}.json'
    with open(caminho_json, 'w', encoding='utf-8') as f:
        json.dump(config_json, f, indent=2, ensure_ascii=False)
    print(f"✅ Configurações JSON: {caminho_json}")

    # 4. Salvar versão mais recente
    caminho_latest = f'modelos/modelo_{nome_ativo}_{estrategia_tag}_latest.pkl'
    joblib.dump(modelo_final, caminho_latest)
    print(f"✅ Versão latest: {caminho_latest}")

    # 5. Backup ZIP
    import zipfile
    zip_path = f'backup/modelo_{nome_ativo}_{estrategia_tag}_{timestamp}.zip'
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        zipf.write(caminho_pkl, os.path.basename(caminho_pkl))
        zipf.write(caminho_json, os.path.basename(caminho_json))
        zipf.write(caminho_simples, os.path.basename(caminho_simples))
    print(f"✅ Backup ZIP: {zip_path}")

    # 6. Google Drive (se disponível)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)

        drive_path = '/content/drive/MyDrive/Colab Notebooks/modelos/'
        os.makedirs(drive_path, exist_ok=True)

        drive_pkl = f'{drive_path}/{nome_base}_{timestamp}.pkl'
        joblib.dump(modelo_final, drive_pkl)
        print(f"\n✅ Modelo salvo no Google Drive: {drive_pkl}")

        drive_latest = f'{drive_path}/modelo_{nome_ativo}_{estrategia_tag}_latest.pkl'
        joblib.dump(modelo_final, drive_latest)
        print(f"✅ Latest no Google Drive: {drive_latest}")
    except:
        print("\n⚠️ Google Drive não disponível")

    return {
        'pkl': caminho_pkl,
        'simples': caminho_simples,
        'json': caminho_json,
        'zip': zip_path,
        'latest': caminho_latest
    }

# ============================================================================
# EXECUÇÃO PRINCIPAL
# ============================================================================

estrategia_desc = descrever_estrategia(HABILITAR_VENDA, USAR_NEUTRO, ALAVANCAGEM_ML, ALAVANCAGEM_BH, CUSTOS)

print("=" * 72)
print(f"  OTIMIZAÇÃO PARA AÇÃO ÚNICA: {NOME_ATIVO} ({TICKER_ESCOLHIDO})")
print("=" * 72)
print(f"  📊 Ação: {NOME_ATIVO}")
print(f"  🎯 Ticker: {TICKER_ESCOLHIDO}")
print(f"  💰 Investimento: R$ {INVESTIMENTO_INICIAL:,.2f}")
print(f"  ⚙️  Estratégia: {estrategia_desc}")
print(f"     • Venda (classe 0): {'✅ Habilitada' if HABILITAR_VENDA else '❌ Desabilitada'}")
print(f"     • Neutro (classe 1): {'✅ Usando' if USAR_NEUTRO else '❌ Não usando'}")
print(f"     • Compra (classe 2): ✅ Sempre habilitada")
print(f"  📈 Alavancagem:")
print(f"     • ML Strategy: {ALAVANCAGEM_ML}x")
print(f"     • Buy & Hold: {ALAVANCAGEM_BH}x")
print(f"  💰 Custos Operacionais (por trade ida e volta):")
print(f"     • Corretagem: {CUSTOS['corretagem']:.3%}")
print(f"     • Slippage: {CUSTOS['slipage']:.3%}")
print(f"     • Emolumentos B3: {CUSTOS['emolumentos']:.3%}")
print(f"     • Taxa Liquidação: {CUSTOS['taxa_liquidacao']:.3%}")
print(f"     • ISS: {CUSTOS['iss']:.3%}")
print(f"     • CUSTO TOTAL: {CUSTOS['custo_total_por_trade']:.3%}")
print(f"  ⚡ Quantis calculados APENAS no treino (sem data leakage)")
print(f"  🔍 Otimizando {len(espaco_parametros)} hiperparâmetros")
print("=" * 72 + "\n")

print("🔍 Iniciando otimização bayesiana...")
print(f"   ⏱️ {TRY} iterações planejadas\n")

res_busca = gp_minimize(
    avaliar_configuracao_acao_unica,
    espaco_parametros,
    n_calls     = TRY,
    random_state= 42,
    verbose     = True
)

print(f"\n{'='*72}")
print("  RESULTADOS DA OTIMIZAÇÃO")
print(f"{'='*72}")
print(f"🎯 Melhor Sharpe (Walk-Forward): {-res_busca.fun:.3f}")
print(f"   Estratégia: {estrategia_desc}")

melhores_params = dict(zip([p.name for p in espaco_parametros], res_busca.x))

print("\n🛠️ PARÂMETROS ÓTIMOS:")
for key, value in melhores_params.items():
    if isinstance(value, float):
        print(f"   {key:18}: {value:.4f}")
    else:
        print(f"   {key:18}: {value}")

TIME_OTIMO = melhores_params['time_period']
ROLL_CURTO_OTIMO = int(melhores_params['roll_curto'])
ROLL_LONGO_OTIMO = int(melhores_params['roll_longo'])

# ============================================================================
# TREINAMENTO FINAL COM PARÂMETROS ÓTIMOS
# ============================================================================

print(f"\n{'='*72}")
print("  TREINAMENTO FINAL COM PARÂMETROS ÓTIMOS")
print(f"{'='*72}")

print(f"\n📥 Baixando dados finais...")
raw = yf.Ticker(TICKER_ESCOLHIDO).history(period=TIME_OTIMO, interval="1d").reset_index()
df = calcular_features_sem_target(raw, ROLL_CURTO_OTIMO, ROLL_LONGO_OTIMO)
print(f"✅ {NOME_ATIVO} ({TICKER_ESCOLHIDO}) — {len(df)} amostras")

# Último split para teste final
tscv = TimeSeriesSplit(n_splits=N_SPLITS_FINAL)
splits = list(tscv.split(df))
idx_treino, idx_teste = splits[-1]

# Separar treino e teste
df_treino = df.iloc[idx_treino].copy()
df_teste = df.iloc[idx_teste].copy()

# Calcular quantis APENAS no treino
driver_fut_treino = df_treino['driver'].shift(-FUT) - df_treino['driver']
limite_venda = driver_fut_treino.quantile(inferior)
limite_compra = driver_fut_treino.quantile(superior)

print(f"\n📏 Limites de classificação (calculados no treino):")
print(f"   Limite Venda: {limite_venda:.4f}")
print(f"   Limite Compra: {limite_compra:.4f}")

# Criar labels no treino
df_treino['Alvo'] = 1
df_treino.loc[driver_fut_treino >= limite_compra, 'Alvo'] = 2
df_treino.loc[driver_fut_treino <= limite_venda, 'Alvo'] = 0

# Criar labels no teste usando MESMOS quantis
driver_fut_teste = df_teste['driver'].shift(-FUT) - df_teste['driver']
df_teste['Alvo'] = 1
df_teste.loc[driver_fut_teste >= limite_compra, 'Alvo'] = 2
df_teste.loc[driver_fut_teste <= limite_venda, 'Alvo'] = 0

# Remover NaNs
df_treino = df_treino.dropna(subset=['Alvo'])
df_teste = df_teste.dropna(subset=['Alvo'])

X_treino = df_treino[FEATURES]
y_treino = df_treino['Alvo']
X_teste = df_teste[FEATURES]
y_teste = df_teste['Alvo']

# Treinar modelo final
print(f"\n🤖 Treinando modelo final...")
print(f"   Treino: {len(X_treino)} amostras")
print(f"   Teste: {len(X_teste)} amostras")

base = XGBClassifier(
    n_estimators    = int(melhores_params['n_estimators']),
    max_depth       = int(melhores_params['max_depth']),
    learning_rate   = melhores_params['learning_rate'],
    min_child_weight= int(melhores_params['min_child_weight']),
    reg_alpha       = melhores_params['reg_alpha'],
    reg_lambda      = melhores_params['reg_lambda'],
    gamma           = melhores_params['gamma'],
    subsample       = melhores_params['subsample'],
    colsample_bytree= melhores_params['colsample_bytree'],
    random_state    = 42,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    verbosity       = 0
)

tscv_calib = TimeSeriesSplit(n_splits=N_SPLITS_FINAL)
modelo_final = CalibratedClassifierCV(base, method='isotonic', cv=tscv_calib)
modelo_final.fit(X_treino, y_treino)

# Previsões
pred_treino = modelo_final.predict(X_treino)
pred_teste = modelo_final.predict(X_teste)
probas_teste = modelo_final.predict_proba(X_teste)

# Gerar sinais de trading conforme configuração
sinal_trading = gerar_sinais_trading(pred_teste, HABILITAR_VENDA, USAR_NEUTRO)

# Métricas de classificação
acc_treino = (pred_treino == y_treino).mean()
acc_teste = (pred_teste == y_teste).mean()
logloss_teste = log_loss(y_teste, probas_teste)
logloss_treino = log_loss(y_treino, modelo_final.predict_proba(X_treino))

# ============================================================================
# BACKTEST FINAL COM CUSTOS (VERSÃO CORRIGIDA)
# ============================================================================

print(f"\n{'='*72}")
print("  BACKTEST FINAL (COM CUSTOS OPERACIONAIS)")
print(f"{'='*72}")

# Preparar dados do teste
testes = df_teste.copy().reset_index(drop=True)
testes['Sinal'] = sinal_trading
testes['Classe'] = pred_teste
testes['Retorno_Real'] = testes['Close'].pct_change()

# APLICAR ALAVANCAGEM NO RETORNO DA ESTRATÉGIA
testes['Retorno_ML'] = testes['Sinal'].shift(1) * testes['Retorno_Real'] * ALAVANCAGEM_ML
testes['Retorno_BH'] = testes['Retorno_Real'] * ALAVANCAGEM_BH

# Aplicar custos operacionais
testes = aplicar_custos_operacionais(testes, CUSTOS)

# Remover NaNs dos retornos
testes_validos = testes.dropna(subset=['Retorno_ML']).copy()

retornos_ml = testes_validos['Retorno_ML']
retornos_ml_bruto = testes_validos['Retorno_ML_Bruto']
retornos_bh = testes_validos['Retorno_BH']

# Métricas financeiras
ret_acum_ml = (1 + retornos_ml).prod() - 1
ret_acum_ml_bruto = (1 + retornos_ml_bruto).prod() - 1
ret_acum_bh = (1 + retornos_bh).prod() - 1
excesso_retorno = ret_acum_ml - ret_acum_bh
impacto_custos = ret_acum_ml_bruto - ret_acum_ml

sharpe_ml = (retornos_ml.mean() / retornos_ml.std()) * np.sqrt(252) if retornos_ml.std() > 0 else 0
sharpe_bh = (retornos_bh.mean() / retornos_bh.std()) * np.sqrt(252) if retornos_bh.std() > 0 else 0

# Valor do portfólio COM alavancagem e custos
valor_ml = INVESTIMENTO_INICIAL * (1 + retornos_ml).cumprod()
valor_ml_bruto = INVESTIMENTO_INICIAL * (1 + retornos_ml_bruto).cumprod()
valor_bh = INVESTIMENTO_INICIAL * (1 + retornos_bh).cumprod()

dd_ml = (valor_ml / valor_ml.cummax() - 1).min()
dd_bh = (valor_bh / valor_bh.cummax() - 1).min()

# Custos totais
custo_total = testes_validos['Custo_Operacao'].sum()
custo_total_pct = custo_total / INVESTIMENTO_INICIAL

# Estatísticas de trades
trades = testes_validos['Sinal'].diff().abs().sum() / 2
trades_compra = (testes_validos['Sinal'] == 1).sum()
trades_venda = (testes_validos['Sinal'] == -1).sum()
trades_neutro = (testes_validos['Sinal'] == 0).sum()

trades_retornos = retornos_ml[retornos_ml != 0]
win_rate = (trades_retornos > 0).mean() if len(trades_retornos) > 0 else 0

# Taxa de sinais
taxa_sinais = (testes_validos['Sinal'] != 0).mean()

# Distribuição de classes
dist_classes = {
    'Venda (0)': (testes_validos['Classe'] == 0).mean(),
    'Neutro (1)': (testes_validos['Classe'] == 1).mean(),
    'Compra (2)': (testes_validos['Classe'] == 2).mean()
}

print(f"\n⚙️  ESTRATÉGIA: {estrategia_desc}")
print(f"{'='*50}")
print(f"  💰 ANÁLISE DE CUSTOS:")
print(f"{'='*50}")
print(f"  Custo Total no Período: R${custo_total:,.2f}")
print(f"  Custo Total (%): {custo_total_pct:.2%}")
print(f"  Custo Médio por Trade: R${custo_total/trades if trades > 0 else 0:,.2f}")
print(f"  Impacto no Retorno: {impacto_custos:.2%}")
print(f"  Retorno Bruto (sem custos): {ret_acum_ml_bruto:.1%}")
print(f"  Retorno Líquido (com custos): {ret_acum_ml:.1%}")

print(f"\n{'='*50}")
print(f"  📊 PERFORMANCE DA ESTRATÉGIA ML:")
print(f"{'='*50}")
print(f"  {'Métrica':35} {'ML Strategy':>12} {'Buy & Hold':>12}")
print(f"  {'-'*50}")
print(f"  {'Capital Final':35} R${valor_ml.iloc[-1]:>11,.2f} R${valor_bh.iloc[-1]:>11,.2f}")
print(f"  {'Retorno Total':35} {ret_acum_ml:>11.1%} {ret_acum_bh:>11.1%}")
print(f"  {'Excesso de Retorno':35} {excesso_retorno:>11.1%}")
print(f"  {'Sharpe Ratio':35} {sharpe_ml:>11.2f} {sharpe_bh:>11.2f}")
print(f"  {'Máximo Drawdown':35} {dd_ml:>11.1%} {dd_bh:>11.1%}")
print(f"  {'Lucro/Prejuízo':35} R${valor_ml.iloc[-1] - INVESTIMENTO_INICIAL:>11,.2f} R${valor_bh.iloc[-1] - INVESTIMENTO_INICIAL:>11,.2f}")

print(f"\n📈 ESTATÍSTICAS DE TRADES:")
print(f"{'='*50}")
print(f"  Total de Trades: {trades:.0f}")
print(f"  Compras: {trades_compra} | Vendas: {trades_venda} | Neutro: {trades_neutro}")
print(f"  Taxa de Sinais: {taxa_sinais:.1%}")
print(f"  Win Rate: {win_rate:.1%}")

print(f"\n🎯 MÉTRICAS DE CLASSIFICAÇÃO:")
print(f"{'='*50}")
print(f"  Acurácia Treino: {acc_treino:.1%}")
print(f"  Acurácia Teste: {acc_teste:.1%}")
print(f"  Gap Acurácia: {acc_treino - acc_teste:.1%}")
print(f"  LogLoss Treino: {logloss_treino:.3f}")
print(f"  LogLoss Teste: {logloss_teste:.3f}")

# ============================================================================
# GRÁFICOS
# ============================================================================

fig = plt.figure(figsize=(20, 16))
fig.patch.set_facecolor('#0E1117')
gs = gridspec.GridSpec(4, 2, figure=fig, wspace=0.25, hspace=0.40)

PANEL_BG = '#161B22'
TEXTO = '#E5E7EB'

# --- Painel 1: Evolução Patrimonial ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(PANEL_BG)
ax1.plot(valor_ml.index, valor_ml.values, color='#2F9FD2', lw=2.5,
         label=f'ML Líquido (Sharpe: {sharpe_ml:.2f})')
ax1.plot(valor_ml_bruto.index, valor_ml_bruto.values, color='#22C55E', lw=1.5, ls=':',
         label=f'ML Bruto (Sem custos)')
ax1.plot(valor_bh.index, valor_bh.values, color='#6B7280', lw=1.5, ls='--',
         label=f'Buy & Hold (Sharpe: {sharpe_bh:.2f})')
ax1.axhline(y=INVESTIMENTO_INICIAL, color='white', lw=0.5, ls=':', alpha=0.5)
ax1.set_title(f'{NOME_ATIVO} ({TICKER_ESCOLHIDO}) - Evolução Patrimonial\n{estrategia_desc}',
              color=TEXTO, fontsize=11, fontweight='bold')
ax1.tick_params(colors=TEXTO, labelsize=8)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
ax1.grid(True, alpha=0.1)
ax1.legend(facecolor=PANEL_BG, labelcolor=TEXTO, fontsize=8)

# --- Painel 2: Drawdown ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor(PANEL_BG)
dd_ml_serie = (valor_ml / valor_ml.cummax() - 1) * 100
dd_bh_serie = (valor_bh / valor_bh.cummax() - 1) * 100
ax2.fill_between(range(len(dd_ml_serie)), dd_ml_serie.values, 0,
                 color='#EF4444', alpha=0.4, label=f'ML (Max DD: {dd_ml:.1%})')
ax2.fill_between(range(len(dd_bh_serie)), dd_bh_serie.values, 0,
                 color='#6B7280', alpha=0.2, label=f'B&H (Max DD: {dd_bh:.1%})')
ax2.set_title('Drawdown (%)', color=TEXTO, fontsize=11, fontweight='bold')
ax2.tick_params(colors=TEXTO, labelsize=8)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax2.grid(True, alpha=0.1)
ax2.legend(facecolor=PANEL_BG, labelcolor=TEXTO, fontsize=8)

# --- Painel 3: Distribuição de Sinais ---
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_facecolor(PANEL_BG)
sinais_dist = [trades_venda, trades_neutro, trades_compra]
cores_sinais = ['#EF4444', '#6B7280', '#22C55E']
labels_sinais = ['Venda', 'Neutro', 'Compra']
bars = ax3.bar(labels_sinais, sinais_dist, color=cores_sinais, alpha=0.8, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, sinais_dist):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', va='bottom', color=TEXTO, fontsize=9)
ax3.set_title(f'Distribuição de Sinais de Trading\n({estrategia_desc})',
              color=TEXTO, fontsize=11, fontweight='bold')
ax3.tick_params(colors=TEXTO, labelsize=9)
ax3.set_ylabel('Número de Dias', color=TEXTO)
ax3.grid(True, alpha=0.1, axis='y')

# --- Painel 4: Impacto dos Custos ---
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_facecolor(PANEL_BG)
custos_diarios = testes_validos['Custo_Operacao'].cumsum()
ax4.fill_between(range(len(custos_diarios)), custos_diarios.values, 0,
                 color='#F59E0B', alpha=0.6, label=f'Custos Acumulados: R${custo_total:,.2f}')
ax4.set_title(f'Custos Operacionais Acumulados\nImpacto Total: {impacto_custos:.2%}',
              color=TEXTO, fontsize=11, fontweight='bold')
ax4.tick_params(colors=TEXTO, labelsize=8)
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
ax4.grid(True, alpha=0.1)
ax4.legend(facecolor=PANEL_BG, labelcolor=TEXTO, fontsize=8)

# --- Painel 5: Convergência da Otimização ---
ax5 = fig.add_subplot(gs[2, 0])
ax5.set_facecolor(PANEL_BG)
sharpe_por_call = [-v for v in res_busca.func_vals]
ax5.plot(range(1, len(sharpe_por_call) + 1), sharpe_por_call,
         color='#F59E0B', lw=2, marker='o', markersize=3, alpha=0.7)
ax5.axhline(y=-res_busca.fun, color='#22C55E', lw=1.5, ls='--',
            label=f'Melhor Sharpe: {-res_busca.fun:.2f}')
ax5.set_title('Convergência da Otimização Bayesiana', color=TEXTO, fontsize=11, fontweight='bold')
ax5.set_xlabel('Iteração', color=TEXTO, fontsize=8)
ax5.set_ylabel('Sharpe Ratio', color=TEXTO, fontsize=8)
ax5.tick_params(colors=TEXTO, labelsize=8)
ax5.grid(True, alpha=0.1)
ax5.legend(facecolor=PANEL_BG, labelcolor=TEXTO, fontsize=8)

# --- Painel 6: Distribuição de Classes ---
ax6 = fig.add_subplot(gs[2, 1])
ax6.set_facecolor(PANEL_BG)
classes_dist = [dist_classes['Venda (0)'], dist_classes['Neutro (1)'], dist_classes['Compra (2)']]
cores_classes = ['#EF4444', '#6B7280', '#22C55E']
wedges, texts, autotexts = ax6.pie(classes_dist, labels=labels_sinais, autopct='%1.1f%%',
                                     colors=cores_classes, startangle=90,
                                     textprops={'color': TEXTO, 'fontsize': 9})
ax6.set_title('Distribuição das Classes (Alvo Ternário)', color=TEXTO, fontsize=11, fontweight='bold')

# --- Painel 7: Retornos com/sem custos ---
ax7 = fig.add_subplot(gs[3, 0])
ax7.set_facecolor(PANEL_BG)
ax7.hist(retornos_ml_bruto, bins=50, alpha=0.5, color='#22C55E', label=f'Bruto (Média: {retornos_ml_bruto.mean():.3%})')
ax7.hist(retornos_ml, bins=50, alpha=0.5, color='#2F9FD2', label=f'Líquido (Média: {retornos_ml.mean():.3%})')
ax7.axvline(x=0, color='white', lw=0.5, ls='-', alpha=0.5)
ax7.set_title('Distribuição dos Retornos (Bruto vs Líquido)', color=TEXTO, fontsize=11, fontweight='bold')
ax7.tick_params(colors=TEXTO, labelsize=8)
ax7.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
ax7.grid(True, alpha=0.1, axis='y')
ax7.legend(facecolor=PANEL_BG, labelcolor=TEXTO, fontsize=8)

# --- Painel 8: Break-even dos Custos ---
ax8 = fig.add_subplot(gs[3, 1])
ax8.set_facecolor(PANEL_BG)

# Calcular trades necessários para break-even
if trades > 0:
    custo_por_trade = custo_total / trades
    retorno_por_trade = (ret_acum_ml_bruto * INVESTIMENTO_INICIAL) / trades

    categorias = ['Retorno Bruto\npor Trade', 'Custo\npor Trade', 'Retorno Líquido\npor Trade']
    valores = [retorno_por_trade, custo_por_trade, retorno_por_trade - custo_por_trade]
    cores = ['#22C55E', '#EF4444', '#2F9FD2']

    bars = ax8.bar(categorias, valores, color=cores, alpha=0.8)
    for bar, val in zip(bars, valores):
        ax8.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'R${val:,.2f}', ha='center', va='bottom', color=TEXTO, fontsize=9)

    ax8.set_title(f'Análise por Trade (Total: {trades:.0f} trades)', color=TEXTO, fontsize=11, fontweight='bold')
    ax8.tick_params(colors=TEXTO, labelsize=8)
    ax8.grid(True, alpha=0.1, axis='y')

plt.suptitle(f'Análise Completa com Custos - {NOME_ATIVO} ({TICKER_ESCOLHIDO})\n{estrategia_desc}',
             color=TEXTO, fontsize=14, fontweight='bold', y=0.98)

estrategia_tag = ""
if HABILITAR_VENDA and USAR_NEUTRO:
    estrategia_tag = "COMPLETO"
elif HABILITAR_VENDA and not USAR_NEUTRO:
    estrategia_tag = "COMPRA_VENDA"
elif not HABILITAR_VENDA and USAR_NEUTRO:
    estrategia_tag = "COMPRA_NEUTRO"
else:
    estrategia_tag = "APENAS_COMPRA"

estrategia_tag += f"_ALAV{ALAVANCAGEM_ML}x_CUSTOS"

# plt.savefig(f'outputs/analise_{NOME_ATIVO}_{TICKER_ESCOLHIDO}_{estrategia_tag}.png',
#             dpi=150, bbox_inches='tight', facecolor='#0E1117')
# plt.show()
# print(f"\n✅ Gráfico salvo em outputs/analise_{NOME_ATIVO}_{TICKER_ESCOLHIDO}_{estrategia_tag}.png")

# ============================================================================
# EXPORTAÇÃO DO MODELO FINAL
# ============================================================================

print(f"\n{'='*72}")
print("  EXPORTAÇÃO DO MODELO FINAL")
print(f"{'='*72}")

modelo_export = {
    'modelo': modelo_final,
    'ticker': TICKER_ESCOLHIDO,
    'nome_ativo': NOME_ATIVO,
    'estrategia': {
        'habilitar_venda': HABILITAR_VENDA,
        'usar_neutro': USAR_NEUTRO,
        'alavancagem_ml': ALAVANCAGEM_ML,
        'alavancagem_bh': ALAVANCAGEM_BH,
        'custos': CUSTOS,
        'descricao': estrategia_desc
    },
    'configuracao': {
        'time_period': TIME_OTIMO,
        'roll_curto': ROLL_CURTO_OTIMO,
        'roll_longo': ROLL_LONGO_OTIMO,
        'features': FEATURES,
        'hiperparametros': melhores_params,
        'limites_classes': {
            'limite_venda': float(limite_venda),
            'limite_compra': float(limite_compra)
        },
        'alvo': 'ternario',
        'classes': {'venda': 0, 'neutro': 1, 'compra': 2},
        'quantis_no_treino': True
    },
    'metricas_backtest': {
        'sharpe_ml': float(sharpe_ml),
        'sharpe_bh': float(sharpe_bh),
        'ret_acum_ml': float(ret_acum_ml),
        'ret_acum_ml_bruto': float(ret_acum_ml_bruto),
        'ret_acum_bh': float(ret_acum_bh),
        'excesso_retorno': float(excesso_retorno),
        'impacto_custos': float(impacto_custos),
        'custo_total': float(custo_total),
        'dd_ml': float(dd_ml),
        'dd_bh': float(dd_bh),
        'acc_treino': float(acc_treino),
        'acc_teste': float(acc_teste),
        'win_rate': float(win_rate),
        'taxa_sinais': float(taxa_sinais),
        'total_trades': float(trades),
        'compras': int(trades_compra),
        'vendas': int(trades_venda),
        'neutros': int(trades_neutro)
    }
}

# Salvar modelo
caminhos = salvar_modelo_acao_unica(modelo_export, TICKER_ESCOLHIDO, NOME_ATIVO)

# ============================================================================
# RESUMO FINAL
# ============================================================================

print(f"\n{'='*72}")
print("  RESUMO FINAL")
print(f"{'='*72}")
print(f"  📊 Ativo analisado: {NOME_ATIVO} ({TICKER_ESCOLHIDO})")
print(f"  ⚙️  Estratégia: {estrategia_desc}")
print(f"  🔍 Dimensões otimizadas: {len(espaco_parametros)}")
print(f"  📅 Período ótimo: {TIME_OTIMO}")
print(f"  🎯 Alvo: TERNÁRIO (Venda/Neutro/Compra)")
print(f"  ⚡ QUARTIS CALCULADOS APENAS NO TREINO (sem data leakage)")
print(f"  🎯 Sharpe ML: {sharpe_ml:.2f} | Sharpe B&H: {sharpe_bh:.2f}")
print(f"  💰 Retorno ML Bruto: {ret_acum_ml_bruto:.1%} | Líquido: {ret_acum_ml:.1%} | B&H: {ret_acum_bh:.1%}")
print(f"  💸 Impacto dos Custos: {impacto_custos:.2%} (R${custo_total:,.2f})")
print(f"  📊 Excesso de Retorno: {excesso_retorno:.1%}")
print(f"  🎯 Acurácia: {acc_teste:.1%} | Win Rate: {win_rate:.1%}")
print(f"  📈 Trades: {trades:.0f} (C:{trades_compra} V:{trades_venda} N:{trades_neutro})")

# Diagnóstico de overfitting
gap_acc = acc_treino - acc_teste
print(f"\n{'='*72}")
print("  DIAGNÓSTICO DE OVERFITTING")
print(f"{'='*72}")
if gap_acc > 0.15:
    status = "🔴 SEVERO - Alta degradação"
elif gap_acc > 0.08:
    status = "🟠 MODERADO - Atenção necessária"
elif gap_acc > 0.03:
    status = "🟡 LEVE - Aceitável"
elif gap_acc < 0:
    status = "🔵 UNDERFITTING - Teste melhor que treino"
else:
    status = "✅ SAUDÁVEL - Performance consistente"

print(f"  Gap de Acurácia: {gap_acc:.1%}")
print(f"  Status: {status}")

# Aviso sobre custos
print(f"\n{'='*72}")
print("  💰 ANÁLISE DE CUSTOS")
print(f"{'='*72}")
print(f"  Custo Total: R${custo_total:,.2f} ({custo_total_pct:.2%} do capital)")
print(f"  Custo por Trade: R${custo_total/trades if trades > 0 else 0:,.2f}")
print(f"  Break-even por trade: R${custo_total/trades if trades > 0 else 0:.2f}")
print(f"  Retorno Líquido por Trade: R${((ret_acum_ml * INVESTIMENTO_INICIAL) / trades) if trades > 0 else 0:,.2f}")

# Aviso sobre alavancagem
if ALAVANCAGEM_ML > 1:
    print(f"\n{'='*72}")
    print("  ⚠️  AVISO DE ALAVANCAGEM")
    print(f"{'='*72}")
    print(f"  Você está usando alavancagem de {ALAVANCAGEM_ML}x")
    print(f"  Lembre-se: Alavancagem amplifica ganhos E perdas!")
    print(f"  Drawdown Máximo com alavancagem: {dd_ml:.1%}")

print(f"\n{'='*72}")
print("  📦 MODELO SALVO COM SUCESSO!")
print(f"{'='*72}")
print(f"\nPara carregar o modelo:")
print(f"  import joblib")
print(f"  modelo = joblib.load('{caminhos['latest']}')")
print(f"\nPara fazer previsões:")
print(f"  pred = modelo['modelo'].predict(X)")
print(f"  → 0 = VENDA, 1 = NEUTRO (HOLD), 2 = COMPRA")
print(f"\nConfiguração da estratégia:")
print(f"  • Venda: {'✅ Habilitada' if HABILITAR_VENDA else '❌ Desabilitada'}")
print(f"  • Neutro: {'✅ Usando' if USAR_NEUTRO else '❌ Não usando'}")
print(f"  • Alavancagem ML: {ALAVANCAGEM_ML}x")
print(f"  • Alavancagem B&H: {ALAVANCAGEM_BH}x")
print(f"  • Custo por Trade: {CUSTOS['custo_total_por_trade']:.3%}")
print(f"\n⚠️ IMPORTANTE: Para novas previsões, use os limites salvos:")
print(f"  Limite Venda: {limite_venda:.4f}")
print(f"  Limite Compra: {limite_compra:.4f}")

print(f"\n✅ Processo completo para {NOME_ATIVO} finalizado com sucesso!")
print(f"   Os resultados já incluem todos os custos operacionais realistas.")